# Ex - Linear Regression: House Price Prediction (with Feature Engineering)

### Introduction:

This exercise uses **scikit-learn's `LinearRegression`** directly — no need to write the optimizer
by hand this time. Instead, all of our energy goes into the skills that actually separate a
mediocre model from a good one in real practice: cleaning data in the RIGHT order, choosing
features by evidence (correlation), engineering new ones, avoiding redundancy between them, and
evaluating honestly.

**Dataset:** real house sale records for **King County, Washington** (which includes Seattle) —
21,613 home sales from 2014–2015, each with its price, size, condition, location, and age.
Widely used as *the* classic real-world regression dataset. Source: public GitHub mirror of the
original Kaggle "House Sales in King County, USA" dataset.

**Goal:** predict `price`. We'll build THREE models:
1. **Simple** — just `sqft_living`
2. **Model A** — a handful of RAW columns, chosen because they correlate with price
3. **Model B** — brand-new ENGINEERED features, built on top of Model A's columns — but with any
   raw column that fed into an engineered feature **removed**, so we never feed the model the same
   information twice under two different names

Notice the ORDER of the steps below — it follows a standard preprocessing pipeline: clean the data
completely (missing values → duplicates → known errors → **outliers**) BEFORE you do anything that
depends on the data being clean, like correlation-based feature selection or feature engineering.
Doing it out of order (e.g. computing correlations before removing outliers) would mean your
feature choices are influenced by exactly the extreme, unusual rows you're trying to ignore.

Work through the steps in order — later steps depend on variables created in earlier ones.

### Step 1. Import the necessary libraries
(`numpy`, `pandas`, `matplotlib`, from scikit-learn: `train_test_split`, `LinearRegression`, `MinMaxScaler`, `r2_score`, `mean_squared_error`, `mean_absolute_percentage_error`, and from `statsmodels`: `variance_inflation_factor`)

### Step 2. Load `kc_house_data.csv` into a variable called `houses`. Print its shape and the first 5 rows.

### Step 3. Look at the data types and general info (`.info()`).

### Step 4. Get summary statistics for every numeric column (`.describe()`).

### Step 5. Check every column for missing values. Since `price` is our TARGET, we can't fill in a guess for a missing price — that would mean training the model on numbers we invented. Drop any rows with a missing `price` instead.

### Step 6. Check the `id` column for duplicates. Investigate: are these truly duplicate ROWS (an error), or something else? Look at a couple of the repeated `id`s and their `date` column before deciding what to do.

### Step 7. Investigate the `bedrooms` column with `.value_counts().sort_index()`. One value looks like an obvious data-entry error, not a real house. Find it, look at that row's other columns (sqft, bathrooms) to confirm it's a typo, and fix it (hint: it was very likely meant to be a 3).

### Step 8. Fix outliers — NOW, as part of cleaning, before any correlation analysis or feature engineering touches this data.

Detect price outliers using the IQR method, computed SEPARATELY for each `grade` group (a $2,000,000 sale is very different for a grade-13 luxury build vs. a grade-5 fixer-upper). Print the bounds and outlier count per grade, then remove them.

### Step 9. NOW do the EDA: a scatter plot of `sqft_living` vs `price` (on the cleaned data), and the correlation of EVERY raw numeric column with `price`, sorted by strength. We'll use this list in the next step.

## 📊 Model A — pick raw features by evidence, not by guessing

### Step 10. Choose Model A's features FROM the correlation list in Step 9 — but not blindly.

Two rules:
1. **Keep only columns with |correlation| ≥ 0.25** with price (a common rule-of-thumb cutoff for "worth including").
2. **Watch out for columns that are just PIECES of another column.** `sqft_above` and `sqft_basement` both correlate with price — but `sqft_living = sqft_above + sqft_basement` EXACTLY (verify this yourself!). Including all three would mean feeding the model the same size information three times. Keep the combined `sqft_living` and drop its two parts.

We're also deliberately leaving `yr_built`, `yr_renovated`, `sqft_basement`, `lat`, `long`, and `sqft_living15` out of Model A, even though some of them are decently correlated — we're saving those columns as raw material for feature engineering in the next section.

👀 Notice: `waterfront` was one of the stronger raw correlations BEFORE we removed outliers in Step 8 — many waterfront homes are extremely expensive, so a lot of them got trimmed as per-grade outliers. Its correlation on the now-cleaned data is much weaker. This is exactly why outlier removal has to come before feature selection — do it the other way around and you'd be choosing features based on rows you're about to throw away.

### Step 11. Do we need to ENCODE anything before modeling? Check the data type of every Model A feature.

## 🛠️ Feature Engineering

Now let's build NEW columns out of the raw material we deliberately held back — `yr_built`, `yr_renovated`, `sqft_basement`, `lat`/`long`, and `sqft_living15`.

### Step 12. `sale_year` and `house_age`
`date` is a text string like `"20141013T000000"` — extract the first 4 characters as `sale_year` (as an integer), then engineer `house_age` = `sale_year` − `yr_built`.

### Step 13. `was_renovated` flag
`yr_renovated` is 0 for houses that were never renovated (0 is a placeholder here, not a real year!). Engineer a simple 0/1 flag `was_renovated` instead of using the raw year directly.

### Step 14. `has_basement` flag
Same idea: `sqft_basement` is 0 for houses with no basement. Engineer a clean 0/1 flag.

### Step 15. `bed_bath_ratio`
Engineer `bed_bath_ratio` = `bedrooms` / `bathrooms` — a single number capturing the LAYOUT of a house (lots of bedrooms but few bathrooms feels very different from a balanced layout), instead of two separate numbers. Watch out: a few houses have `bathrooms == 0`, which would divide by zero — replace those with the median ratio instead.

⚠️ This one uses `bedrooms` and `bathrooms` as its source — both are currently IN Model A. Keep that in mind for Step 19.

### Step 16. `distance_to_downtown_km` — a geospatial feature!
We have `lat`/`long` for every house. Engineer the straight-line distance (in km) from each house to downtown Seattle (47.6062° N, -122.3321° W), using the **Haversine formula** (the standard formula for distance between two points on a sphere, given their latitude/longitude):

```
a = sin²(Δlat/2) + cos(lat1)·cos(lat2)·sin²(Δlong/2)
distance = 2 · R · asin(√a)      (R = 6371 km, Earth's radius)
```

### Step 17. `living_area_vs_neighbors`
`sqft_living15` is the average living area of a house's 15 nearest neighbors. Engineer `living_area_vs_neighbors` = `sqft_living` − `sqft_living15` — a positive number means this house is BIGGER than the houses around it (often a premium), negative means smaller.

⚠️ This one uses `sqft_living` as its source — and `sqft_living` is currently IN Model A. Keep that in mind for Step 19.

### Step 18. ⚠️ The trap: why NOT engineer `price_per_sqft` as a model INPUT
It's tempting to engineer `price_per_sqft = price / sqft_living` as a feature. In a markdown cell, explain in 1-2 sentences why this would be a serious mistake for a model that's trying to PREDICT `price`. (No code needed — this step is a trap to reason through, not to write.)

### Step 19. Did the engineering actually work? Print the correlation of every ENGINEERED feature (`house_age`, `was_renovated`, `has_basement`, `bed_bath_ratio`, `distance_to_downtown_km`, `living_area_vs_neighbors`) with `price`, sorted by strength.

## 📊 Model B — engineered features, WITHOUT their own sources

### Step 20. Build Model B's feature list: start from Model A's features, **remove any raw column that was used as a SOURCE to build an engineered feature** (per the ⚠️ notes in Steps 15 and 17: that's `bedrooms`, `bathrooms`, and `sqft_living`), then add all six engineered features. If we kept both a raw column AND something built directly from it, the model would be told the same information twice under two different names — that's a form of multicollinearity.

### Step 21. Split into training (80%) and test (20%) sets with `train_test_split`, `random_state=1`. Call the results `train_df` and `test_df`.

### Step 22. Fit the SIMPLE model with scikit-learn: `price ~ sqft_living`. Print the intercept and coefficient as a readable equation.

### Step 23. Fit **Model A** (the correlation-selected raw features from Step 10) with scikit-learn. Print every coefficient.

### Step 24. Fit **Model B** (Step 20's feature list — sources removed) with scikit-learn. Print every coefficient. Compare the signs and sizes to Model A — do they look more sensible now that nothing is duplicated?

### Step 25. Double-check Model B for leftover multicollinearity using **VIF (Variance Inflation Factor)** — a formal statistical measure (not just a guess). A rule of thumb: VIF > 5 is concerning, VIF > 10 is a real problem. Print the VIF for every Model B feature.

## 📏 Feature scaling — not needed to FIT the model, but needed to COMPARE it

### Step 26. scikit-learn's `LinearRegression` finds the exact best-fit line no matter what scale the input columns are in — it doesn't need normalization to work correctly (that requirement is specific to gradient-descent-based optimizers, which we're not using here). BUT the RAW coefficients from Step 24 aren't fairly comparable to each other: `grade` only ranges 0–13, while `living_area_vs_neighbors` ranges over thousands of square feet — so their coefficients live on totally different scales and can't be used to rank "what matters most."

Normalize Model B's features into [0, 1] with `MinMaxScaler`, refit `LinearRegression` on the normalized values, and sort the new coefficients by size — THIS ranking is a fair comparison of feature importance.

### 📎 Formula reference — you'll need these for the evaluation steps

For any set of predictions, with `n` = number of rows, `actual` = real values, `predicted` = your
model's guesses, and `mean(actual)` = the average of the real values:

| Name | Formula | Plain-English meaning |
|---|---|---|
| **SST** (Total Sum of Squares) | `Σ (actual − mean(actual))²` | How spread out the real values are, ignoring the model entirely |
| **SSE** (Sum of Squared Errors) | `Σ (actual − predicted)²` | How wrong the model's predictions are (a.k.a. residual sum of squares) |
| **SSR** (Regression Sum of Squares) | `Σ (predicted − mean(actual))²` | How much of that spread the model explains |
| **Identity** | `SST = SSR + SSE` | Total spread = explained spread + unexplained spread |
| **R²** | `1 − SSE/SST` (equivalently `SSR/SST`) | Fraction of the spread the model explains (1.0 = perfect) |
| **MSE** | `SSE / n` | Average squared mistake |
| **RMSE** | `√MSE` | Average mistake, back in the SAME units as price ($) |
| **MAPE** | `mean(\|actual − predicted\| / \|actual\|) × 100` | Average mistake as a percentage of the real value |

⚠️ `SST = SSR + SSE` holds **exactly** on the data a model was fit on (the training set) — that's
a mathematical guarantee of least-squares fitting. It's usually only *approximately* true on a
held-out test set.

### Step 27. On the TRAINING set, compute SST, SSR, and SSE **by hand** for Model B (using the ORIGINAL, real-unit `model_b` from Step 24 — Step 26's normalized version was only for the importance comparison), and confirm `SST ≈ SSR + SSE`.

### Step 28. On the TEST set, write your OWN function to compute R², MSE, RMSE, and MAPE (using the formulas from the reference above — don't just call `r2_score`), then use it to evaluate all THREE models (Simple, Model A, Model B) side by side. Double-check your numbers against scikit-learn's `r2_score` / `mean_squared_error` / `mean_absolute_percentage_error` for Model B.

### Step 29. Reflection (answer in this markdown cell, no code needed):
1. Model B has FEWER raw columns than Model A (three were removed as sources) but gained six engineered ones. Did it still win on R² and RMSE? By how much?
2. Look at the VIF values from Step 25 — are they all comfortably low now? What were the worst-offending columns before we removed `bedrooms`, `bathrooms`, and `sqft_living` (think back to Steps 15 and 17's ⚠️ warnings)?
3. From Step 26's normalized comparison, which feature has the single biggest effect on price, in a fair scale? Does that match what you'd have guessed from the raw-unit coefficients in Step 24?
4. `waterfront` was one of the strongest raw correlations before Step 8's outlier removal, but didn't make the cut afterward. Do you agree with leaving it out, or would you keep it anyway based on real-world knowledge that waterfront property carries a big premium? What does that tell you about blindly trusting a correlation threshold?
5. What's the danger of engineering a feature FROM the target, like `price_per_sqft` (Step 18)? Can you think of another feature in this dataset that would have the same problem?